In [ ]:
"""
simons_nb.ipynb

figures for simons meeting

Author: Stellina X. Ao
Created: 2026-09-21
Last Modified: 2026-09-21
Python Version: 3.11.14
"""


import scienceplots  # noqa: F401
import shutup
import matplotlib.pyplot as plt

%load_ext autoreload
%autoreload 2

# pretty plots
plt.style.use(["nature"])
plt.rcParams["figure.dpi"] = 200
%matplotlib widget
%config InlineBackend.print_figure_kwargs = {'bbox_inches':None}

# suppress warnings :-)
shutup.please()

In [ ]:
subj_id = "MR82"
sess_id = "20251027_152036"

# population

In [ ]:
from sg.models import Encoder, StrategyEncoder

encoder = Encoder(subj_id, sess_id, norm=True)
encoder.verify()

encoder_mb = StrategyEncoder(subj_id, sess_id, norm=True, strategy_filter="mb")
encoder_mf = StrategyEncoder(subj_id, sess_id, norm=True, strategy_filter="mf")

encoder_mb.verify()
encoder_mb.fit_encoder()

encoder_mf.verify()
encoder_mf.fit_encoder()

## r2

In [ ]:
# significance test between mb and mf

In [ ]:
# non-parametric significance test between mb/mf explained variance, bonferroni for region
from scipy.stats import wilcoxon, false_discovery_control

alpha = 0.05

ps = [
    wilcoxon(
        x=encoder_mb.scores["encoder"][encoder_mb.reg_idxs[reg]],
        y=encoder_mf.scores["encoder"][encoder_mf.reg_idxs[reg]],
    ).pvalue
    for reg in encoder.regions
]
ps_corr = false_discovery_control(ps)
ps_corr = {reg: ps_corr[i] for i, reg in enumerate(encoder.regions)}

print("sig diff r2 between mb and mf?")
for reg in encoder.regions:
    print(reg, ps_corr[reg] < alpha, f"({ps_corr[reg]:.3f})")

In [ ]:
# distro figs
from core.viz import plot_kdes
from utils.colors import colors_region
from utils.viz_utils import save_fig
from utils.paths import FIGURES_DIR

fig_full, ax_full = plot_kdes(
    data={
        reg: encoder.scores["encoder"][encoder.reg_idxs[reg]] for reg in encoder.regions
    },
    line_kwargs={reg: {"color": c} for reg, c in colors_region.items()},
    xlim=[-0.2, 1.0],
    label=r"$r^2$, full",
)
ax_full.axvline(x=0, color="#333333", linestyle="--")

fig_strat, ax_strat = plot_kdes(
    data={
        reg: encoder_mb.scores["encoder"][encoder_mb.reg_idxs[reg]]
        - encoder_mf.scores["encoder"][encoder_mf.reg_idxs[reg]]
        for reg in encoder.regions
    },
    line_kwargs={reg: {"color": c} for reg, c in colors_region.items()},
    xlim=[-1.0, 1.0],
    label=r"$r^2$, (mb-mf)",
)
ax_strat.axvline(x=0, color="#333333", linestyle="--")
ax_strat.set_title(f"DMS (p={ps_corr['DMS']:.3f}), DLS (p={ps_corr['DLS']:.3f})")

for fext in ["svg", "png"]:
    save_fig(
        fig_full,
        FIGURES_DIR / "r2" / "distros" / subj_id / sess_id,
        fname=f"r2_distro_full-{subj_id}_{sess_id}.{fext}",
    )
    save_fig(
        fig_strat,
        FIGURES_DIR / "r2" / "distros" / subj_id / sess_id,
        fname=f"r2_distro_strat-{subj_id}_{sess_id}.{fext}",
    )

## beta weight

In [ ]:
for regr in encoder.tv_keys:
    # sig
    ps = [
        wilcoxon(
            x=encoder_mb.encoder_weights[
                encoder_mb.reg_idxs[reg], encoder_mb.tv_idxs[regr]
            ],
            y=encoder_mf.encoder_weights[
                encoder_mf.reg_idxs[reg], encoder_mf.tv_idxs[regr]
            ],
        ).pvalue
        for reg in encoder.regions
    ]
    ps_corr = false_discovery_control(ps)
    ps_corr = {reg: ps_corr[i] for i, reg in enumerate(encoder.regions)}

    regr_tex = regr.replace("_", r"\_")
    fig_full, ax_full = plot_kdes(
        data={
            reg: encoder.encoder_weights[encoder.reg_idxs[reg], encoder.tv_idxs[regr]]
            for reg in encoder.regions
        },
        line_kwargs={reg: {"color": c} for reg, c in colors_region.items()},
        xlim=[-1.0, 1.0],
        label=rf"$\beta_{{\mathrm{{{regr_tex}}}}}$, full",
    )
    ax_full.axvline(x=0, color="#333333", linestyle="--")

    fig_strat, ax_strat = plot_kdes(
        data={
            reg: encoder_mb.encoder_weights[
                encoder_mb.reg_idxs[reg], encoder_mb.tv_idxs[regr]
            ]
            - encoder_mf.encoder_weights[
                encoder_mf.reg_idxs[reg], encoder_mf.tv_idxs[regr]
            ]
            for reg in encoder.regions
        },
        line_kwargs={reg: {"color": c} for reg, c in colors_region.items()},
        xlim=[-1.0, 1.0],
        label=rf"$\beta_{{\mathrm{{{regr_tex}}}}}$, (mb-mf)",
    )
    ax_strat.axvline(x=0, color="#333333", linestyle="--")
    ax_strat.set_title(f"DMS (p={ps_corr['DMS']:.3f}), DLS (p={ps_corr['DLS']:.3f})")

    for fext in ["svg", "png"]:
        save_fig(
            fig_full,
            FIGURES_DIR / "bweight" / "distros" / subj_id / sess_id,
            fname=f"{regr}-bweight_distro_full-{subj_id}_{sess_id}.{fext}",
        )
        save_fig(
            fig_strat,
            FIGURES_DIR / "bweight" / "distros" / subj_id / sess_id,
            fname=f"{regr}-bweight_distro_strat-{subj_id}_{sess_id}.{fext}",
        )

In [ ]:
ps_corr = {regr: {} for regr in encoder.tv_keys}
alpha = 0.05

for regr in encoder.tv_keys:
    ps = [
        wilcoxon(
            x=encoder_mb.encoder_weights[
                encoder_mb.reg_idxs[reg], encoder_mb.tv_idxs[regr]
            ],
            y=encoder_mf.encoder_weights[
                encoder_mf.reg_idxs[reg], encoder_mf.tv_idxs[regr]
            ],
        ).pvalue
        for reg in encoder.regions
    ]
    ps_corr_ = false_discovery_control(ps)
    ps_corr[regr] = {reg: ps_corr_[i] for i, reg in enumerate(encoder.regions)}

print("sig diff bweight between mb and mf?")
for regr in encoder.tv_keys:
    print(regr)
    for reg in encoder.regions:
        print(f"\t {reg}, {ps_corr[regr][reg] < alpha}, ({ps_corr[regr][reg]:.3f})")

## cvr2, dr2
be aware that interaction terms (but not trials from block switch) are included

In [ ]:
from sg.models import ShuffledEncoder

se = ShuffledEncoder(
    subj_id,
    sess_id,
    enc_class=Encoder,
    tv_keys=[
        "response",
        "rewarded",
        "block_side",
        "response_prev",
        "rewarded_prev",
    ],
    add_interaction=True,
)
se.get_cvr2_all()
se.get_dr2_all()

In [ ]:
se_mb = ShuffledEncoder(
    subj_id,
    sess_id,
    enc_class=StrategyEncoder,
    strategy_filter="mb",
    tv_keys=[
        "response",
        "rewarded",
        "block_side",
        "response_prev",
        "rewarded_prev",
    ],
    add_interaction=True,
)

se_mf = ShuffledEncoder(
    subj_id,
    sess_id,
    enc_class=StrategyEncoder,
    strategy_filter="mf",
    tv_keys=[
        "response",
        "rewarded",
        "block_side",
        "response_prev",
        "rewarded_prev",
    ],
    add_interaction=True,
)
print("mb, cvr2")
se_mb.get_cvr2_all()
print("mb, dr2")
se_mb.get_dr2_all()

print("mf, cvr2")
se_mf.get_cvr2_all()
print("mf, dr2")
se_mf.get_dr2_all()

### cvr2

In [ ]:
for regr in encoder.tv_keys:
    # sig
    ps = [
        wilcoxon(
            x=se_mb.cvr2_unit[regr][:, encoder_mb.reg_idxs[reg]].mean(axis=0),
            y=se_mf.cvr2_unit[regr][:, encoder_mf.reg_idxs[reg]].mean(axis=0),
        ).pvalue
        for reg in encoder.regions
    ]
    ps_corr = false_discovery_control(ps)
    ps_corr = {reg: ps_corr[i] for i, reg in enumerate(encoder.regions)}

    regr_tex = regr.replace("_", r"\_")
    fig_full, ax_full = plot_kdes(
        data={
            reg: se.cvr2_unit[regr][:, encoder.reg_idxs[reg]].mean(axis=0)
            for reg in encoder.regions
        },
        line_kwargs={reg: {"color": c} for reg, c in colors_region.items()},
        xlim=[-0.2, 0.8],
        label=rf"$\text{{cv }} r^2_{{\mathrm{{{regr_tex}}}}}$, full",
    )
    ax_full.axvline(x=0, color="#333333", linestyle="--")

    fig_strat, ax_strat = plot_kdes(
        data={
            reg: se_mb.cvr2_unit[regr][:, encoder_mb.reg_idxs[reg]].mean(axis=0)
            - se_mf.cvr2_unit[regr][:, encoder_mf.reg_idxs[reg]].mean(axis=0)
            for reg in encoder.regions
        },
        line_kwargs={reg: {"color": c} for reg, c in colors_region.items()},
        xlim=[-0.8, 0.8],
        label=rf"$\text{{cv }} r^2_{{\mathrm{{{regr_tex}}}}}$, (mb-mf)",
    )
    ax_strat.axvline(x=0, color="#333333", linestyle="--")
    ax_strat.set_title(f"DMS (p={ps_corr['DMS']:.3f}), DLS (p={ps_corr['DLS']:.3f})")

    for fext in ["svg", "png"]:
        save_fig(
            fig_full,
            FIGURES_DIR / "cv_d_r2" / "cv" / "distros" / subj_id / sess_id,
            fname=f"{regr}-cvr2_distro_full-{subj_id}_{sess_id}.{fext}",
        )
        save_fig(
            fig_strat,
            FIGURES_DIR / "cv_d_r2" / "cv" / "distros" / subj_id / sess_id,
            fname=f"{regr}-cvr2_distro_strat-{subj_id}_{sess_id}.{fext}",
        )

In [ ]:
ps_corr = {regr: {} for regr in encoder.tv_keys}
alpha = 0.05

for regr in encoder.tv_keys:
    ps = [
        wilcoxon(
            x=se_mb.cvr2_unit[regr][:, encoder_mb.reg_idxs[reg]].mean(axis=0),
            y=se_mf.cvr2_unit[regr][:, encoder_mf.reg_idxs[reg]].mean(axis=0),
        ).pvalue
        for reg in encoder.regions
    ]
    ps_corr_ = false_discovery_control(ps)
    ps_corr[regr] = {reg: ps_corr_[i] for i, reg in enumerate(encoder.regions)}

print("sig diff cvr2 between mb and mf?")
for regr in encoder.tv_keys:
    print(regr)
    for reg in encoder.regions:
        print(f"\t {reg}, {ps_corr[regr][reg] < alpha}, ({ps_corr[regr][reg]:.3f})")

### delta r2

In [ ]:
for regr in encoder.tv_keys:
    # sig
    ps = [
        wilcoxon(
            x=se_mb.dr2_unit[regr][:, encoder_mb.reg_idxs[reg]].mean(axis=0),
            y=se_mf.dr2_unit[regr][:, encoder_mf.reg_idxs[reg]].mean(axis=0),
        ).pvalue
        for reg in encoder.regions
    ]
    ps_corr = false_discovery_control(ps)
    ps_corr = {reg: ps_corr[i] for i, reg in enumerate(encoder.regions)}

    regr_tex = regr.replace("_", r"\_")
    fig_full, ax_full = plot_kdes(
        data={
            reg: se.dr2_unit[regr][:, encoder.reg_idxs[reg]].mean(axis=0)
            for reg in encoder.regions
        },
        line_kwargs={reg: {"color": c} for reg, c in colors_region.items()},
        xlim=[-0.05, 0.1],
        label=rf"$\text{{cv }} r^2_{{\mathrm{{{regr_tex}}}}}$, full",
    )
    ax_full.axvline(x=0, color="#333333", linestyle="--")

    fig_strat, ax_strat = plot_kdes(
        data={
            reg: se_mb.dr2_unit[regr][:, encoder_mb.reg_idxs[reg]].mean(axis=0)
            - se_mf.dr2_unit[regr][:, encoder_mf.reg_idxs[reg]].mean(axis=0)
            for reg in encoder.regions
        },
        line_kwargs={reg: {"color": c} for reg, c in colors_region.items()},
        xlim=[-0.3, 0.3],
        label=rf"$\text{{cv }} r^2_{{\mathrm{{{regr_tex}}}}}$, (mb-mf)",
    )
    ax_strat.axvline(x=0, color="#333333", linestyle="--")
    ax_strat.set_title(f"DMS (p={ps_corr['DMS']:.3f}), DLS (p={ps_corr['DLS']:.3f})")

    # for fext in ["svg", "png"]:
    #     save_fig(
    #         fig_full,
    #         FIGURES_DIR / "cv_d_r2" / "delta" / "distros" / subj_id / sess_id,
    #         fname=f"{regr}-dr2_distro_full-{subj_id}_{sess_id}.{fext}",
    #     )
    #     save_fig(
    #         fig_strat,
    #         FIGURES_DIR / "cv_d_r2" / "delta" / "distros" / subj_id / sess_id,
    #         fname=f"{regr}-dr2_distro_strat-{subj_id}_{sess_id}.{fext}",
    #     )

In [ ]:
ps_corr = {regr: {} for regr in encoder.tv_keys}
alpha = 0.05

for regr in encoder.tv_keys:
    ps = [
        wilcoxon(
            x=se_mb.dr2_unit[regr][:, encoder_mb.reg_idxs[reg]].mean(axis=0),
            y=se_mf.dr2_unit[regr][:, encoder_mf.reg_idxs[reg]].mean(axis=0),
        ).pvalue
        for reg in encoder.regions
    ]
    ps_corr_ = false_discovery_control(ps)
    ps_corr[regr] = {reg: ps_corr_[i] for i, reg in enumerate(encoder.regions)}

print("sig diff dr2 between mb and mf?")
for regr in encoder.tv_keys:
    print(regr)
    for reg in encoder.regions:
        print(f"\t {reg}, {ps_corr[regr][reg] < alpha}, ({ps_corr[regr][reg]:.3f})")

## sig pie chart

In [ ]:
from sg.models import BootstrapperShuffle as BSS

bss = BSS(
    subj_id,
    sess_id,
    Encoder,
    n=20,
    norm=True,
)
bss.get_ci_idxs()

bss_mb = BSS(
    subj_id,
    sess_id,
    StrategyEncoder,
    strategy_filter="mb",
    n=20,
    norm=True,
)
bss_mb.get_ci_idxs()

bss_mf = BSS(
    subj_id,
    sess_id,
    StrategyEncoder,
    strategy_filter="mf",
    n=20,
    norm=True,
)
bss_mf.get_ci_idxs()

In [ ]:
p_sig = {
    regr: {
        reg: len(bss.ci_idxs_reg[regr][reg]) / len(encoder.psths[reg])
        for reg in encoder.regions
    }
    for regr in encoder.tv_keys
}

p_sig_mb = {
    regr: {
        reg: len(bss_mb.ci_idxs_reg[regr][reg]) / len(encoder.psths[reg])
        for reg in encoder.regions
    }
    for regr in encoder.tv_keys
}

p_sig_mf = {
    regr: {
        reg: len(bss_mf.ci_idxs_reg[regr][reg]) / len(encoder.psths[reg])
        for reg in encoder.regions
    }
    for regr in encoder.tv_keys
}

In [ ]:
from core.viz import plot_grouped_bar_h

fig, ax = plot_grouped_bar_h(
    data=p_sig, ylabel="p(significant)", colors=colors_region, legend=False
)
ax.axvline(x=1.0, color="#333333", linewidth=0.75)

In [ ]:
# maybe because neurons in dms are more likely to encode task variables strongly in only one strategy, when you consider both strategies, those neurons become not significant